# Product-Image Classifier (CNN)

This notebook trains a small CNN to sort product photos into **Apparel**, **Electronics**, and **Home**.

**How to run this:**
1. Open this file in Google Colab (upload it, or once it's on GitHub: File → Open Notebook → GitHub).
2. Go to `Runtime -> Change runtime type -> Hardware accelerator -> GPU`, then save.
3. Run the cells top to bottom, one at a time — read the comments as you go.

We're using **TensorFlow / Keras**: it handles the training loop for you (`model.fit(...)`), so there's less boilerplate to learn before you see results.

In [1]:
# Quick check that Colab actually gave us a GPU.
# If this prints an empty list, go to Runtime -> Change runtime type -> GPU, then re-run this cell.
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Step 1: Get your data into Colab

You've already downloaded a Kaggle e-commerce product image set and organized it on your laptop into:

```
data/raw/
├── train/
│   ├── Apparel/
│   ├── Electronics/
│   └── Home/
└── val/
    ├── Apparel/
    ├── Electronics/
    └── Home/
```

Colab runs on Google's servers, so it can't see files on your laptop — we need to upload them.

**Before running the next cell:** on your laptop, right-click the `raw` folder (inside `data/`) and compress it into a file called `data_raw.zip`.

In [ ]:
from google.colab import files
files.upload()  # select data_raw.zip when prompted — this may take a few minutes depending on dataset size

!mkdir -p data/raw
!unzip -q data_raw.zip -d data/raw
!echo "Done. Folder structure:"
!find data/raw -maxdepth 2 -type d

**Check the output above.** You should see `data/raw/train/Apparel`, `data/raw/train/Electronics`, `data/raw/train/Home`, and the same three under `val`. If the paths look different (e.g. there's an extra `raw` folder nested inside), adjust `DATA_DIR` in the next cell to match.

## Step 2: Load the images

Keras has a built-in helper, `image_dataset_from_directory`, that reads images straight out of folders like ours — it automatically treats each subfolder name (`Apparel`, `Electronics`, `Home`) as a class label. This is exactly why we organized the data that way.

In [ ]:
DATA_DIR = "data/raw"
IMG_SIZE = (160, 160)   # resize every image to this size — keeps things small & fast on Colab
BATCH_SIZE = 32          # how many images the model looks at per training step

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",  # one-hot labels, since we have 3 classes
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

class_names = train_ds.class_names
print("Classes found:", class_names)

## Step 3: Look at what we actually have

Before building anything, it's worth sanity-checking the data — how many images per class (a big imbalance would need fixing), and what the images actually look like.

In [ ]:
import matplotlib.pyplot as plt

# Count images per class in the training set
for i, name in enumerate(class_names):
    count = len(list(__import__("pathlib").Path(f"{DATA_DIR}/train/{name}").glob("*")))
    print(f"{name}: {count} images")

# Show a grid of sample images with their labels
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        label_idx = tf.argmax(labels[i]).numpy()
        plt.title(class_names[label_idx])
        plt.axis("off")

**If the class counts are wildly uneven** (say one class has 5x more images than another), tell me — we'll either trim the bigger classes down or use class weights during training so the model doesn't just learn to guess the biggest class.

## Step 4: Preprocessing

Two things happen here:
- **Normalization** — pixel values start as 0–255; neural networks train better when inputs are small numbers, so we rescale to 0–1.
- **Augmentation** — randomly flip/rotate/zoom training images each epoch, so the model sees slightly different versions each time instead of memorizing the exact same 1,000 photos. This is one of the simplest ways to fight overfitting on a smallish dataset.

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# Cache + prefetch: speeds up training by preparing the next batch while the GPU is busy with the current one
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Preprocessing ready. Data pipeline set up for training.")

## Next step

Run everything above. Check that:
1. `class_names` printed `['Apparel', 'Electronics', 'Home']` (order may vary — that's fine)
2. The sample image grid actually shows sensible photos with correct-looking labels
3. The per-class counts aren't wildly imbalanced

Once that's confirmed, come back and we'll build the actual CNN architecture next.